In [1]:
from google.colab import drive
drive.mount('/content/gdrive/')

Mounted at /content/gdrive/


In [ ]:
!cp -r "/content/gdrive/MyDrive/DATABASES/CelebAspoof/README/metas/" .

In [ ]:
!cp "/content/gdrive/MyDrive/DATABASES/CelebAspoof/face/CelebA_Spoof_Faces.zip" .
!unzip -qq CelebA_Spoof_Faces.zip

In [2]:
import pandas as pd
import numpy as np
import glob
from pathlib import Path
import cv2
import matplotlib.pyplot as plt
import tqdm
import skimage.util as sk_noise
import PIL as pil
import json
from sklearn.model_selection import train_test_split

In [3]:
import os
import shutil

In [ ]:
!rm -r CelebA_Spoof_Faces.zip

In [ ]:
df_train_json = pd.read_json('metas/intra_test/train_label.json').T
df_test_json = pd.read_json('metas/intra_test/test_label.json').T

select_columns = ['full_path_iqa', 'data_type', 'person', 'environment',
                  'illumination_condition', 'spoof_type', 'label', 'label_value']
list_df = []
for i in [df_train_json, df_test_json]:
    df_tmp = i.copy()
    df_tmp['full_path_iqa'] = df_tmp.index
    df_tmp['data_type'] = df_tmp.full_path_iqa.apply(lambda x: x.split('/')[1])
    df_tmp['person'] = df_tmp.full_path_iqa.apply(lambda x: x.split('/')[2])
    df_tmp['environment'] = df_tmp[42]
    df_tmp['illumination_condition'] = df_tmp[41]
    df_tmp['spoof_type'] = df_tmp[40]
    df_tmp['label'] = df_tmp.full_path_iqa.apply(lambda x: x.split('/')[-2])
    df_tmp['label_value'] = df_tmp[43]
    df_tmp = df_tmp[select_columns]
    list_df.append(df_tmp)

df = pd.concat(list_df).reset_index(drop=True)
df.full_path_iqa = df.full_path_iqa.apply(lambda x: x.replace('.png', '.jpg').replace('.jpeg', '.jpg'))
df.head(3)

,full_path_iqa,data_type,person,environment,illumination_condition,spoof_type,label,label_value
0,Data/train/2623/live/000000.jpg,train,2623,0,0,0,live,0
1,Data/train/5489/spoof/000001.jpg,train,5489,1,1,6,spoof,1
2,Data/train/7149/spoof/000002.jpg,train,7149,1,1,7,spoof,1


In [8]:
df_train = df[df.data_type=='train']
df_test = df[df.data_type=='test']

train_val, _ = train_test_split(df_train, stratify=df_train[['environment',
                                                   'illumination_condition',
                                                   'spoof_type',
                                                   'label']], train_size=10000/df_train.shape[0], random_state=42)

train, val = train_test_split(train_val, stratify=train_val[['environment',
                                                   'illumination_condition',
                                                   'spoof_type',
                                                   'label']], train_size=0.85, random_state=42)

train.shape, val.shape, train.shape[0]+val.shape[0]

((8502, 8), (1498, 8), 10000)

In [ ]:
val['data_type'] = 'val'
df_rd = pd.concat([train, val])

In [ ]:
df_rd.to_csv('/content/gdrive/MyDrive/DATABASES/CelebAspoof/face/ceas_faces_intradataset_rd_trainval_min.csv', index=False)

In [ ]:
!rm -r Data/
for idx, row in df_rd.iterrows():
    full_path = row['full_path_iqa']

    dir_path_new = '/'.join(full_path.split('/')[:-1])
    Path(f"{dir_path_new}").mkdir(parents=True, exist_ok=True)

    shutil.copy('CelebA_Spoof_Faces/'+full_path, full_path)


rm: cannot remove 'Data/': No such file or directory


In [ ]:
!zip -r "CelebA_Spoof_Faces_rd.zip" Data/ ceas_iqa.csv -q

In [ ]:
!cp "CelebA_Spoof_Faces_rd.zip" "/content/gdrive/MyDrive/DATABASES/CelebAspoof/face/"

In [ ]:
df_train = df[df.data_type=='train']
df_test = df[df.data_type=='test']
train, _ = train_test_split(df_train, stratify=df_train[['environment',
                                                   'illumination_condition',
                                                   'spoof_type',
                                                   'label']], train_size = 1700/df_train.shape[0], random_state=42)
_, test = train_test_split(df_test, stratify=df_test[['environment',
                                                   'illumination_condition',
                                                   'spoof_type',
                                                   'label']], test_size = 300/df_test.shape[0], random_state=42)
train.shape, test.shape, train.shape[0]+test.shape[0]

((1700, 8), (300, 8), 2000)

In [ ]:
df_for_iqa = pd.concat([train, test])
df_for_iqa.sample(3)

,full_path_iqa,data_type,person,environment,illumination_condition,spoof_type,label,label_value
373421,Data/train/9289/spoof/373421.jpg,train,9289,1,1,10,spoof,1
482601,Data/train/1431/live/482601.jpg,train,1431,0,0,0,live,0
542774,Data/test/6945/spoof/542774.jpg,test,6945,2,1,4,spoof,1


In [ ]:
!rm -r fas_iqa/
fas_iqa_dataset = []
for idx, row in df_for_iqa.iterrows():
    full_path = row['full_path_iqa']
    frame = row['full_path_iqa'].split('/')[-1]

    client = row['person']
    label = 'real' if row['label'] == 'live' else 'attack'
    data_type = row['data_type']

    scene = f"client{('0'*6+client)[-6:]}_Env{row['environment']}_Ilum{row['illumination_condition']}_Spt{row['spoof_type']}"

    iqa_class = 'original'

    dir_path_new = f"fas_iqa/{data_type}/{iqa_class}/{label}/{scene}/"
    Path(f"{dir_path_new}").mkdir(parents=True, exist_ok=True)
    full_path_new = f"{dir_path_new}/{frame}"

    shutil.copy('CelebA_Spoof_Faces/'+full_path, full_path_new)
    if os.path.isfile(f"{full_path_new}"):
        fas_iqa_dataset.append({

            'frame': frame,
            'scene': scene,
            'client': client,
            'label': label,
            'data_type': data_type,
            'full_path': full_path_new,

            'iqa': 'original',
        })

df_fas_iqa = pd.DataFrame(fas_iqa_dataset)
df_fas_iqa.sample(3)

,frame,scene,client,label,data_type,full_path,iqa
1936,523608.jpg,client009239_Env1_Ilum1_Spt9,9239,attack,test,fas_iqa/test/original/attack/client009239_Env1...,original
1927,524804.jpg,client005192_Env1_Ilum1_Spt5,5192,attack,test,fas_iqa/test/original/attack/client005192_Env1...,original
660,258868.jpg,client008973_Env0_Ilum0_Spt0,8973,real,train,fas_iqa/train/original/real/client008973_Env0_...,original


In [ ]:
df_fas_iqa.to_csv('ceas_iqa.csv', index=False)

In [ ]:
!zip -r "ceas_iqa.zip" fas_iqa/ ceas_iqa.csv -q

In [ ]:
!cp "ceas_iqa.zip" "/content/gdrive/MyDrive/DATABASES/CelebAspoof/face/"

In [ ]:
df_fas_iqa.shape

(2000, 7)

END